In [5]:
import igl
from pathlib import Path
import csv
import math
import numpy as np

# Use absolute path
base_path = Path("/Users/aa13586/Desktop/symmetric-dirichlet")

uv_path = base_path / "data" / "closed-Myles-dijkstra-01-11" / "filigree100k_param.obj"
v3d, uv, _, f, fuv, _ = igl.readOBJ(uv_path)
print(len(v3d), len(uv), len(f), len(fuv))

149942 174325 300140 300140


In [2]:
import re

def read_vertex_list(path):
    verts = []
    with open(path, newline='') as csvfile:
        reader = csv.reader(csvfile)
        for row in reader:
            if not row:
                continue
            text = " ".join(row).strip()
            if text == "":
                continue
            for tok in re.split(r'[,\s]+', text):
                if not tok:
                    continue
                try:
                    verts.append(int(tok))
                except ValueError:
                    # skip non-integer tokens
                    continue
    return verts


verts_csv = base_path / "output" / "max_hessian_vertices.csv"
input_indices = read_vertex_list(verts_csv)
print(max(input_indices), len(v3d), len(uv))

172771 149942 174325


In [3]:
def triangle_area(v0, v1, v2):
    a = np.linalg.norm(v1 - v0)
    b = np.linalg.norm(v2 - v1)
    c = np.linalg.norm(v0 - v2)
    s = (a + b + c) / 2.0
    area = max(s * (s - a) * (s - b) * (s - c), 1e-10)**0.5
    inradius = area / s
    circumradius = (a * b * c) / (4.0 * area)
    return area, circumradius / inradius
# build adjacency: faces incident to each vertex
faces_uv = []
for f_0 in range(len(fuv)):
    if fuv[f_0][0] in input_indices or fuv[f_0][1] in input_indices or fuv[f_0][2] in input_indices:
        faces_uv.append(f_0)
print(faces_uv[:5], len(faces_uv))

[41240, 41251, 41253, 41257, 42871] 352


In [ ]:
import csv

def uv_triangle_areas(uv, F):
    """Compute areas of triangles in UV space"""
    u0 = uv[F[:, 0], :]
    u1 = uv[F[:, 1], :]
    u2 = uv[F[:, 2], :]
    cross_z = (u1[:,0] - u0[:,0]) * (u2[:,1] - u0[:,1]) - (u1[:,1] - u0[:,1]) * (u2[:,0] - u0[:,0])
    areas = 0.5 * np.abs(cross_z)
    return areas

def compute_face_metrics(uv, faces_uv, input_indices):
    """
    Compute metrics for each face in faces_uv:
    - face index
    - area (UV)
    - aspect ratio (circumradius / inradius)
    - min edge length
    - length of edge containing vertex from input_indices
    """
    rows = []
    
    for face_idx in faces_uv:
        v0_idx, v1_idx, v2_idx = fuv[face_idx]
        v0 = uv[v0_idx]
        v1 = uv[v1_idx]
        v2 = uv[v2_idx]
        
        # Edge lengths in UV space
        e0 = np.linalg.norm(v1 - v2)  # edge opposite v0
        e1 = np.linalg.norm(v2 - v0)  # edge opposite v1
        e2 = np.linalg.norm(v0 - v1)  # edge opposite v2
        
        # Area (Heron's formula)
        s = (e0 + e1 + e2) / 2.0
        area_sq = s * (s - e0) * (s - e1) * (s - e2)
        area = np.sqrt(max(area_sq, 1e-10))
        print(area)
        
        # Aspect ratio (circumradius / inradius)
        inradius = area / s if s > 0 else 1e-10
        circumradius = (e0 * e1 * e2) / (4.0 * area) if area > 0 else 1e10
        aspect_ratio = circumradius / inradius if inradius > 0 else 1e10
        
        # Min edge
        min_edge = min(e0, e1, e2)
        max_edge = max(e0, e1, e2)

        # Find edge length containing vertex from input_indices
        edge_with_special_vertex = None
        for vi in [v0_idx, v1_idx, v2_idx]:
            if vi in input_indices:
                if vi == v0_idx:
                    # v0 is special; edges are e1 and e2
                    edge_with_special_vertex = max(e1, e2)
                elif vi == v1_idx:
                    # v1 is special; edges are e0 and e2
                    edge_with_special_vertex = max(e0, e2)
                else:  # vi == v2_idx
                    # v2 is special; edges are e0 and e1
                    edge_with_special_vertex = max(e0, e1)
                break
        
        if edge_with_special_vertex is None:
            edge_with_special_vertex = 0.0  # no special vertex in face
        
        rows.append([
            face_idx,  # tuple of indices
            area,
            aspect_ratio,
            min_edge,
            max_edge,
            edge_with_special_vertex
        ])
    
    return rows

# Compute metrics
metrics = compute_face_metrics(uv, faces_uv, set(input_indices))

# Write to CSV
output_csv = base_path / "output" / "face_metrics.csv"
output_csv.parent.mkdir(parents=True, exist_ok=True)

with open(output_csv, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["face_indices", "area_uv", "aspect_ratio", "min_edge", "max_edge"])
    for row in metrics:
        writer.writerow([row[0], row[1], row[2], row[3], row[4]])

print(f"Wrote {len(metrics)} face metrics to {output_csv}")

1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-05
1e-0